In [1]:
# ================================================================
# 🏭 SMART FACTORY AI - ADVANCED PREDICTIVE MAINTENANCE
# Google Colab | No Streamlit
# ================================================================

# ---------------- INSTALL ----------------
!pip -q install reportlab scikit-learn matplotlib pandas numpy ipywidgets openpyxl

# ---------------- IMPORTS ----------------
import os
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, Image, PageBreak
)
from reportlab.lib.units import inch

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files

warnings.filterwarnings("ignore")


# ================================================================
# 1. CREATE SYNTHETIC TRAINING DATA
# ================================================================

np.random.seed(42)

N = 12000

temp = np.clip(np.random.normal(72, 13, N), 30, 130)
vib = np.clip(np.random.normal(4.2, 1.6, N), 0, 12)
pres = np.clip(np.random.normal(100, 16, N), 50, 160)
rpm = np.clip(np.random.normal(1500, 280, N), 500, 2500)
trq = np.clip(np.random.normal(52, 14, N), 10, 100)

# Continuous risk score
risk_score = (
    0.35 * np.clip((temp - 70) / 35, 0, 1.5)
    + 0.30 * np.clip((vib - 3.5) / 5, 0, 1.5)
    + 0.10 * np.clip((pres - 100) / 45, 0, 1)
    + 0.15 * np.clip((rpm - 1500) / 800, 0, 1)
    + 0.10 * np.clip((trq - 55) / 40, 0, 1)
)

# Failure label
failure = (
    (
        ((temp > 92).astype(int)) +
        ((vib > 6.2).astype(int)) +
        ((pres > 135).astype(int)) +
        ((rpm > 2100).astype(int)) +
        ((trq > 80).astype(int))
    ) >= 2
).astype(int)

train_df = pd.DataFrame({
    "Temperature": temp,
    "Vibration": vib,
    "Pressure": pres,
    "RPM": rpm,
    "Torque": trq,
    "Failure": failure
})

FEATURES = [
    "Temperature",
    "Vibration",
    "Pressure",
    "RPM",
    "Torque"
]

X = train_df[FEATURES]
y = train_df["Failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=250,
    max_depth=14,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

test_prediction = model.predict(X_test)
model_accuracy = accuracy_score(
    y_test,
    test_prediction
) * 100


# ================================================================
# 2. STORAGE
# ================================================================

records = []


# ================================================================
# 3. SENSOR LIMITS
# ================================================================

LIMITS = {
    "Temperature": 92,
    "Vibration": 6.2,
    "Pressure": 135,
    "RPM": 2100,
    "Torque": 80
}


# ================================================================
# 4. HEADER
# ================================================================

display(HTML("""
<div style="
background:linear-gradient(135deg,#081c2c,#123b55,#176b87);
color:white;
padding:28px;
border-radius:18px;
text-align:center;
box-shadow:0 5px 18px rgba(0,0,0,.25);
">
<h1 style="margin:0;">🏭 SMART FACTORY AI</h1>
<h3>Advanced Predictive Maintenance System</h3>
<p>Real-time sensor analysis • AI failure prediction • Maintenance intelligence</p>
</div>
"""))


# ================================================================
# 5. INPUT FORM
# ================================================================

machine = widgets.Text(
    value="Machine-01",
    description="🏭 Machine:",
    layout=widgets.Layout(width="450px")
)

operator = widgets.Text(
    value="Operator",
    description="👤 Operator:",
    layout=widgets.Layout(width="450px")
)

temperature = widgets.FloatSlider(
    value=70,
    min=30,
    max=130,
    step=0.5,
    description="🌡️ Temperature:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="550px")
)

vibration = widgets.FloatSlider(
    value=4,
    min=0,
    max=12,
    step=0.1,
    description="〰️ Vibration:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="550px")
)

pressure = widgets.FloatSlider(
    value=100,
    min=50,
    max=160,
    step=1,
    description="💨 Pressure:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="550px")
)

rpm = widgets.IntSlider(
    value=1500,
    min=500,
    max=2500,
    step=10,
    description="⚙️ RPM:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="550px")
)

torque = widgets.FloatSlider(
    value=50,
    min=10,
    max=100,
    step=1,
    description="🔩 Torque:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="550px")
)


# ================================================================
# 6. BUTTONS
# ================================================================

predict_btn = widgets.Button(
    description="🤖 Predict & Save",
    button_style="success",
    layout=widgets.Layout(width="180px", height="45px")
)

demo_btn = widgets.Button(
    description="🧪 Add Demo Data",
    button_style="primary",
    layout=widgets.Layout(width="170px", height="45px")
)

report_btn = widgets.Button(
    description="📄 Download PDF",
    button_style="info",
    layout=widgets.Layout(width="170px", height="45px")
)

csv_btn = widgets.Button(
    description="📊 Download CSV",
    button_style="warning",
    layout=widgets.Layout(width="170px", height="45px")
)

reset_btn = widgets.Button(
    description="🗑️ Reset",
    button_style="danger",
    layout=widgets.Layout(width="140px", height="45px")
)

output = widgets.Output()


# ================================================================
# 7. AI ANALYSIS FUNCTION
# ================================================================

def analyze(t, v, p, r, tq):

    data = pd.DataFrame(
        [[t, v, p, r, tq]],
        columns=FEATURES
    )

    ai_probability = (
        model.predict_proba(data)[0][1] * 100
    )

    # Rule-based sensor severity
    abnormalities = []

    if t > LIMITS["Temperature"]:
        abnormalities.append(
            f"High temperature ({t:.1f} °C)"
        )

    if v > LIMITS["Vibration"]:
        abnormalities.append(
            f"High vibration ({v:.1f} mm/s)"
        )

    if p > LIMITS["Pressure"]:
        abnormalities.append(
            f"High pressure ({p:.1f} PSI)"
        )

    if r > LIMITS["RPM"]:
        abnormalities.append(
            f"High RPM ({r})"
        )

    if tq > LIMITS["Torque"]:
        abnormalities.append(
            f"High torque ({tq:.1f} Nm)"
        )

    # Combine AI + sensor abnormalities
    if len(abnormalities) >= 2:
        final_risk = max(
            ai_probability,
            min(100, ai_probability + 10)
        )
    else:
        final_risk = ai_probability

    health = max(
        0,
        min(100, 100 - final_risk)
    )

    if final_risk >= 70:
        status = "CRITICAL"
        emoji = "🔴"

    elif final_risk >= 35:
        status = "WARNING"
        emoji = "🟡"

    else:
        status = "HEALTHY"
        emoji = "🟢"

    if status == "CRITICAL":
        recommendation = (
            "Immediate inspection required. "
            "Consider controlled shutdown before continuing operation."
        )

    elif status == "WARNING":
        recommendation = (
            "Schedule preventive maintenance and "
            "closely monitor abnormal parameters."
        )

    else:
        recommendation = (
            "Machine parameters are currently within "
            "the monitored operating range."
        )

    if not abnormalities:
        abnormalities = [
            "No abnormal sensor threshold detected."
        ]

    return (
        final_risk,
        health,
        status,
        emoji,
        abnormalities,
        recommendation
    )


# ================================================================
# 8. DASHBOARD RENDER
# ================================================================

def render_dashboard():

    if len(records) == 0:
        return

    df = pd.DataFrame(records)

    # Latest machine
    latest = df.iloc[-1]

    status = latest["Status"]

    status_color = {
        "HEALTHY": "#16a085",
        "WARNING": "#f39c12",
        "CRITICAL": "#c0392b"
    }[status]

    display(HTML(f"""
    <div style="
    background:{status_color};
    color:white;
    padding:22px;
    border-radius:15px;
    text-align:center;
    margin:15px 0;
    ">
    <h2>{latest["Status"]}</h2>
    <h3>{latest["Machine"]}</h3>
    <p style="font-size:21px;">
    Failure Risk: <b>{latest["Failure Risk (%)"]:.1f}%</b>
    &nbsp;&nbsp; | &nbsp;&nbsp;
    Health: <b>{latest["Health Score"]:.1f}/100</b>
    </p>
    </div>
    """))

    # KPI cards
    display(HTML(f"""
    <div style="display:flex;gap:12px;margin:15px 0;">

    <div style="flex:1;background:#ecf0f1;padding:15px;
    border-radius:12px;text-align:center;">
    <h4>Total Entries</h4>
    <h2>{len(df)}</h2>
    </div>

    <div style="flex:1;background:#d5f5e3;padding:15px;
    border-radius:12px;text-align:center;">
    <h4>Healthy</h4>
    <h2>{(df["Status"]=="HEALTHY").sum()}</h2>
    </div>

    <div style="flex:1;background:#fdebd0;padding:15px;
    border-radius:12px;text-align:center;">
    <h4>Warning</h4>
    <h2>{(df["Status"]=="WARNING").sum()}</h2>
    </div>

    <div style="flex:1;background:#fadbd8;padding:15px;
    border-radius:12px;text-align:center;">
    <h4>Critical</h4>
    <h2>{(df["Status"]=="CRITICAL").sum()}</h2>
    </div>

    </div>
    """))

    print("\n📋 MACHINE HISTORY")
    display(df)


# ================================================================
# 9. PREDICT & SAVE
# ================================================================

def predict_and_save(btn):

    with output:

        clear_output(wait=True)

        t = temperature.value
        v = vibration.value
        p = pressure.value
        r = rpm.value
        tq = torque.value

        (
            risk,
            health,
            status,
            emoji,
            abnormalities,
            recommendation
        ) = analyze(
            t, v, p, r, tq
        )

        timestamp = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        records.append({
            "Timestamp": timestamp,
            "Machine": machine.value,
            "Operator": operator.value,
            "Temperature": t,
            "Vibration": v,
            "Pressure": p,
            "RPM": r,
            "Torque": tq,
            "Failure Risk (%)": round(risk, 2),
            "Health Score": round(health, 2),
            "Status": status
        })

        print("=" * 75)
        print("🤖 AI PREDICTION COMPLETED")
        print("=" * 75)

        print(f"\nMachine       : {machine.value}")
        print(f"Timestamp     : {timestamp}")
        print(f"Failure Risk  : {risk:.1f}%")
        print(f"Health Score  : {health:.1f}/100")
        print(f"Status        : {emoji} {status}")

        print("\n⚠️ SENSOR ANALYSIS")
        print("-" * 75)

        for item in abnormalities:
            print("•", item)

        print("\n💡 MAINTENANCE RECOMMENDATION")
        print("-" * 75)
        print(recommendation)

        render_dashboard()

        # Current sensor graph
        plt.figure(figsize=(12,5))

        names = [
            "Temperature",
            "Vibration",
            "Pressure",
            "RPM",
            "Torque"
        ]

        values = [
            t, v, p, r, tq
        ]

        bars = plt.bar(
            names,
            values,
            color=[
                "#e74c3c",
                "#f39c12",
                "#3498db",
                "#9b59b6",
                "#2ecc71"
            ]
        )

        plt.title(
            f"{machine.value} - Current Sensor Readings",
            fontsize=16,
            fontweight="bold"
        )

        plt.ylabel("Value")
        plt.grid(axis="y", alpha=.25)

        for bar, value in zip(
            bars,
            values
        ):
            plt.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height(),
                f"{value:.1f}",
                ha="center",
                va="bottom"
            )

        plt.tight_layout()
        plt.show()
        plt.close()


# ================================================================
# 10. DEMO DATA
# ================================================================

def add_demo(btn):

    demo_values = [
        ("Machine-01", 68, 3.5, 98, 1450, 48),
        ("Machine-01", 76, 4.2, 105, 1550, 52),
        ("Machine-01", 86, 5.3, 112, 1750, 62),
        ("Machine-01", 96, 6.8, 125, 1950, 70),
        ("Machine-01", 108, 8.2, 142, 2200, 86),
    ]

    for item in demo_values:

        m, t, v, p, r, tq = item

        risk, health, status, _, _, _ = analyze(
            t, v, p, r, tq
        )

        records.append({
            "Timestamp": datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            "Machine": m,
            "Operator": "Demo",
            "Temperature": t,
            "Vibration": v,
            "Pressure": p,
            "RPM": r,
            "Torque": tq,
            "Failure Risk (%)": round(risk, 2),
            "Health Score": round(health, 2),
            "Status": status
        })

    with output:

        clear_output(wait=True)

        print("🧪 5 DEMO RECORDS ADDED")
        print("=" * 75)

        render_dashboard()

        # Risk trend
        df = pd.DataFrame(records)

        plt.figure(figsize=(11,5))

        plt.plot(
            range(1, len(df)+1),
            df["Failure Risk (%)"],
            marker="o",
            linewidth=2.5,
            color="#e74c3c"
        )

        plt.axhline(
            70,
            color="red",
            linestyle="--",
            label="Critical"
        )

        plt.axhline(
            35,
            color="orange",
            linestyle="--",
            label="Warning"
        )

        plt.ylim(0,100)
        plt.title(
            "AI Failure Risk Trend",
            fontsize=16,
            fontweight="bold"
        )

        plt.xlabel("Entry")
        plt.ylabel("Risk (%)")
        plt.grid(alpha=.25)
        plt.legend()

        plt.tight_layout()
        plt.show()
        plt.close()


# ================================================================
# 11. CSV EXPORT
# ================================================================

def export_csv(btn):

    with output:

        if len(records) == 0:

            clear_output()
            print("⚠️ No data available.")
            return

        df = pd.DataFrame(records)

        filename = (
            "/content/"
            "Smart_Factory_Machine_History.xlsx"
        )

        df.to_excel(
            filename,
            index=False
        )

        print("✅ Excel file generated.")
        print("📥 Download starting...")

        files.download(filename)


# ================================================================
# 12. RESET
# ================================================================

def reset_all(btn):

    records.clear()

    with output:

        clear_output()

        print("=" * 75)
        print("🗑️ ALL DATA RESET")
        print("=" * 75)
        print("System ready for new machine entries.")


# ================================================================
# 13. CREATE PDF GRAPHS
# ================================================================

def create_pdf_graphs(df):

    x = np.arange(
        1,
        len(df)+1
    )

    # ---------------- RISK ----------------

    risk_file = (
        "/content/"
        "pdf_failure_risk.png"
    )

    plt.figure(figsize=(10,5))

    plt.plot(
        x,
        df["Failure Risk (%)"],
        marker="o",
        linewidth=2.5,
        color="#e74c3c"
    )

    plt.axhline(
        70,
        color="red",
        linestyle="--",
        label="Critical"
    )

    plt.axhline(
        35,
        color="orange",
        linestyle="--",
        label="Warning"
    )

    plt.ylim(0,100)
    plt.title(
        "AI Failure Risk Trend",
        fontsize=16,
        fontweight="bold"
    )

    plt.xlabel("Entry")
    plt.ylabel("Failure Risk (%)")
    plt.grid(alpha=.25)
    plt.legend()

    plt.tight_layout()
    plt.savefig(
        risk_file,
        dpi=220,
        bbox_inches="tight"
    )

    plt.close()

    # ---------------- HEALTH ----------------

    health_file = (
        "/content/"
        "pdf_health.png"
    )

    plt.figure(figsize=(10,5))

    plt.plot(
        x,
        df["Health Score"],
        marker="o",
        linewidth=2.5,
        color="#16a085"
    )

    plt.ylim(0,100)

    plt.title(
        "Machine Health Score Trend",
        fontsize=16,
        fontweight="bold"
    )

    plt.xlabel("Entry")
    plt.ylabel("Health Score")
    plt.grid(alpha=.25)

    plt.tight_layout()
    plt.savefig(
        health_file,
        dpi=220,
        bbox_inches="tight"
    )

    plt.close()

    # ---------------- TEMPERATURE ----------------

    temp_file = (
        "/content/"
        "pdf_temperature.png"
    )

    plt.figure(figsize=(10,5))

    plt.plot(
        x,
        df["Temperature"],
        marker="o",
        linewidth=2,
        color="#e67e22"
    )

    plt.axhline(
        92,
        color="red",
        linestyle="--",
        label="High Temperature"
    )

    plt.title(
        "Temperature Trend",
        fontsize=16,
        fontweight="bold"
    )

    plt.xlabel("Entry")
    plt.ylabel("Temperature °C")
    plt.grid(alpha=.25)
    plt.legend()

    plt.tight_layout()
    plt.savefig(
        temp_file,
        dpi=220,
        bbox_inches="tight"
    )

    plt.close()

    # ---------------- VIBRATION ----------------

    vibration_file = (
        "/content/"
        "pdf_vibration.png"
    )

    plt.figure(figsize=(10,5))

    plt.plot(
        x,
        df["Vibration"],
        marker="o",
        linewidth=2,
        color="#8e44ad"
    )

    plt.axhline(
        6.2,
        color="red",
        linestyle="--",
        label="High Vibration"
    )

    plt.title(
        "Vibration Trend",
        fontsize=16,
        fontweight="bold"
    )

    plt.xlabel("Entry")
    plt.ylabel("Vibration mm/s")
    plt.grid(alpha=.25)
    plt.legend()

    plt.tight_layout()
    plt.savefig(
        vibration_file,
        dpi=220,
        bbox_inches="tight"
    )

    plt.close()

    return (
        risk_file,
        health_file,
        temp_file,
        vibration_file
    )


# ================================================================
# 14. PDF GENERATOR
# ================================================================

def generate_pdf(btn):

    with output:

        clear_output(wait=True)

        if len(records) == 0:

            print(
                "⚠️ Pehle machine data add karo."
            )

            return

        df = pd.DataFrame(records)

        (
            risk_file,
            health_file,
            temp_file,
            vibration_file
        ) = create_pdf_graphs(df)

        pdf_file = (
            "/content/"
            "Smart_Factory_AI_Advanced_Report.pdf"
        )

        doc = SimpleDocTemplate(
            pdf_file,
            pagesize=A4,
            rightMargin=38,
            leftMargin=38,
            topMargin=38,
            bottomMargin=38
        )

        styles = getSampleStyleSheet()

        title = ParagraphStyle(
            "title",
            parent=styles["Title"],
            fontSize=22,
            leading=27,
            alignment=TA_CENTER,
            textColor=colors.HexColor("#123b55"),
            spaceAfter=20
        )

        heading = ParagraphStyle(
            "heading",
            parent=styles["Heading2"],
            fontSize=16,
            textColor=colors.HexColor("#176b87"),
            spaceAfter=12
        )

        normal = ParagraphStyle(
            "normal",
            parent=styles["Normal"],
            fontSize=9.5,
            leading=14
        )

        story = []

        # ========================================================
        # PAGE 1 - COVER / SUMMARY
        # ========================================================

        story.append(
            Paragraph(
                "🏭 SMART FACTORY AI",
                title
            )
        )

        story.append(
            Paragraph(
                "Advanced Predictive Maintenance Report",
                title
            )
        )

        story.append(
            Spacer(1,10)
        )

        latest = df.iloc[-1]

        summary = [
            ["Metric", "Latest Result"],
            [
                "Machine",
                str(latest["Machine"])
            ],
            [
                "Status",
                str(latest["Status"])
            ],
            [
                "Failure Risk",
                f'{latest["Failure Risk (%)"]:.1f}%'
            ],
            [
                "Health Score",
                f'{latest["Health Score"]:.1f}/100'
            ],
            [
                "Total Entries",
                str(len(df))
            ],
            [
                "Model Accuracy",
                f"{model_accuracy:.1f}%"
            ]
        ]

        table = Table(
            summary,
            colWidths=[220,220]
        )

        table.setStyle(
            TableStyle([
                (
                    "BACKGROUND",
                    (0,0),
                    (-1,0),
                    colors.HexColor("#123b55")
                ),
                (
                    "TEXTCOLOR",
                    (0,0),
                    (-1,0),
                    colors.white
                ),
                (
                    "FONTNAME",
                    (0,0),
                    (-1,0),
                    "Helvetica-Bold"
                ),
                (
                    "GRID",
                    (0,0),
                    (-1,-1),
                    .5,
                    colors.grey
                ),
                (
                    "ALIGN",
                    (0,0),
                    (-1,-1),
                    "CENTER"
                ),
                (
                    "TOPPADDING",
                    (0,0),
                    (-1,-1),
                    9
                ),
                (
                    "BOTTOMPADDING",
                    (0,0),
                    (-1,-1),
                    9
                )
            ])
        )

        story.append(table)

        story.append(
            Spacer(1,25)
        )

        story.append(
            Paragraph(
                "System Overview",
                heading
            )
        )

        story.append(
            Paragraph(
                "This report summarizes machine sensor data, "
                "AI-based failure probability, health score, "
                "sensor abnormalities and predictive maintenance "
                "indicators generated by the Smart Factory AI system.",
                normal
            )
        )

        # ========================================================
        # PAGE 2 - DATA TABLE
        # ========================================================

        story.append(PageBreak())

        story.append(
            Paragraph(
                "1. Machine Data History",
                title
            )
        )

        headers = [
            "Machine",
            "Temp",
            "Vib",
            "Press",
            "RPM",
            "Torque",
            "Risk",
            "Health",
            "Status"
        ]

        table_data = [headers]

        for _, row in df.iterrows():

            table_data.append([
                str(row["Machine"]),
                f'{row["Temperature"]:.1f}',
                f'{row["Vibration"]:.1f}',
                f'{row["Pressure"]:.1f}',
                f'{row["RPM"]}',
                f'{row["Torque"]:.1f}',
                f'{row["Failure Risk (%)"]:.1f}%',
                f'{row["Health Score"]:.1f}',
                str(row["Status"])
            ])

        data_table = Table(
            table_data,
            repeatRows=1,
            colWidths=[
                55,40,40,45,
                40,45,45,45,55
            ]
        )

        data_table.setStyle(
            TableStyle([
                (
                    "BACKGROUND",
                    (0,0),
                    (-1,0),
                    colors.HexColor("#176b87")
                ),
                (
                    "TEXTCOLOR",
                    (0,0),
                    (-1,0),
                    colors.white
                ),
                (
                    "FONTNAME",
                    (0,0),
                    (-1,0),
                    "Helvetica-Bold"
                ),
                (
                    "FONTSIZE",
                    (0,0),
                    (-1,-1),
                    7
                ),
                (
                    "GRID",
                    (0,0),
                    (-1,-1),
                    .4,
                    colors.grey
                ),
                (
                    "ALIGN",
                    (0,0),
                    (-1,-1),
                    "CENTER"
                ),
                (
                    "TOPPADDING",
                    (0,0),
                    (-1,-1),
                    6
                ),
                (
                    "BOTTOMPADDING",
                    (0,0),
                    (-1,-1),
                    6
                )
            ])
        )

        story.append(data_table)

        # ========================================================
        # PAGE 3 - RISK
        # ========================================================

        story.append(PageBreak())

        story.append(
            Paragraph(
                "2. AI Failure Risk Trend",
                title
            )
        )

        story.append(
            Image(
                risk_file,
                width=7.0*inch,
                height=4.2*inch
            )
        )

        story.append(
            Spacer(1,20)
        )

        story.append(
            Paragraph(
                "Risk interpretation: below 35% = Healthy, "
                "35–69.9% = Warning, 70% or above = Critical.",
                normal
            )
        )

        # ========================================================
        # PAGE 4 - HEALTH
        # ========================================================

        story.append(PageBreak())

        story.append(
            Paragraph(
                "3. Machine Health Trend",
                title
            )
        )

        story.append(
            Image(
                health_file,
                width=7.0*inch,
                height=4.2*inch
            )
        )

        # ========================================================
        # PAGE 5 - TEMPERATURE
        # ========================================================

        story.append(PageBreak())

        story.append(
            Paragraph(
                "4. Temperature Analysis",
                title
            )
        )

        story.append(
            Image(
                temp_file,
                width=7.0*inch,
                height=4.2*inch
            )
        )

        # ========================================================
        # PAGE 6 - VIBRATION
        # ========================================================

        story.append(PageBreak())

        story.append(
            Paragraph(
                "5. Vibration Analysis",
                title
            )
        )

        story.append(
            Image(
                vibration_file,
                width=7.0*inch,
                height=4.2*inch
            )
        )

        # ========================================================
        # PAGE 7 - AI MODEL
        # ========================================================

        story.append(PageBreak())

        story.append(
            Paragraph(
                "6. AI Model & Feature Importance",
                title
            )
        )

        importance = pd.Series(
            model.feature_importances_,
            index=FEATURES
        ).sort_values(
            ascending=False
        )

        importance_data = [
            ["Sensor", "Importance"]
        ]

        for sensor, value in importance.items():

            importance_data.append([
                sensor,
                f"{value*100:.2f}%"
            ])

        importance_table = Table(
            importance_data,
            colWidths=[220,220]
        )

        importance_table.setStyle(
            TableStyle([
                (
                    "BACKGROUND",
                    (0,0),
                    (-1,0),
                    colors.HexColor("#123b55")
                ),
                (
                    "TEXTCOLOR",
                    (0,0),
                    (-1,0),
                    colors.white
                ),
                (
                    "FONTNAME",
                    (0,0),
                    (-1,0),
                    "Helvetica-Bold"
                ),
                (
                    "GRID",
                    (0,0),
                    (-1,-1),
                    .5,
                    colors.grey
                ),
                (
                    "ALIGN",
                    (0,0),
                    (-1,-1),
                    "CENTER"
                ),
                (
                    "TOPPADDING",
                    (0,0),
                    (-1,-1),
                    9
                ),
                (
                    "BOTTOMPADDING",
                    (0,0),
                    (-1,-1),
                    9
                )
            ])
        )

        story.append(
            importance_table
        )

        story.append(
            Spacer(1,20)
        )

        story.append(
            Paragraph(
                f"Random Forest model accuracy on the held-out "
                f"test dataset: <b>{model_accuracy:.1f}%</b>.",
                normal
            )
        )

        story.append(
            Spacer(1,10)
        )

        story.append(
            Paragraph(
                "Note: The demonstration model is trained on "
                "synthetic sensor data. For real industrial use, "
                "the model should be retrained and validated using "
                "actual machine sensor and maintenance/failure records.",
                normal
            )
        )

        # ========================================================
        # BUILD
        # ========================================================

        doc.build(story)

        print("=" * 75)
        print("✅ ADVANCED PDF REPORT READY")
        print("=" * 75)
        print(f"📄 {pdf_file}")
        print("📑 7 separate pages created.")
        print("📥 Download starting...")
        print("=" * 75)

        files.download(pdf_file)


# ================================================================
# 15. CONNECT BUTTONS
# ================================================================

predict_btn.on_click(
    predict_and_save
)

demo_btn.on_click(
    add_demo
)

report_btn.on_click(
    generate_pdf
)

csv_btn.on_click(
    export_csv
)

reset_btn.on_click(
    reset_all
)


# ================================================================
# 16. DISPLAY FORM
# ================================================================

display(HTML("""
<h2 style="color:#123b55;">📝 New Machine Data Entry</h2>
<p>
Enter sensor readings below and click
<b>Predict & Save</b>.
</p>
"""))

display(machine)
display(operator)

display(temperature)
display(vibration)
display(pressure)
display(rpm)
display(torque)

display(
    widgets.HBox([
        predict_btn,
        demo_btn,
        report_btn,
        csv_btn,
        reset_btn
    ])
)

display(output)


# ================================================================
# 17. SYSTEM STATUS
# ================================================================

display(HTML(f"""
<div style="
margin-top:20px;
padding:15px;
background:#eaf2f8;
border-radius:12px;
color:#123b55;
">
<b>🟢 System Ready</b><br>
AI Model: Random Forest<br>
Training Samples: {N:,}<br>
Test Accuracy: {model_accuracy:.1f}%<br>
Status: Ready for new machine data
</div>
"""))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 40.3 MB/s eta 0:00:00


Text(value='Machine-01', description='🏭 Machine:', layout=Layout(width='450px'))

Text(value='Operator', description='👤 Operator:', layout=Layout(width='450px'))

FloatSlider(value=70.0, description='🌡️ Temperature:', layout=Layout(width='550px'), max=130.0, min=30.0, step…

FloatSlider(value=4.0, description='〰️ Vibration:', layout=Layout(width='550px'), max=12.0, style=SliderStyle(…

FloatSlider(value=100.0, description='💨 Pressure:', layout=Layout(width='550px'), max=160.0, min=50.0, step=1.…

IntSlider(value=1500, description='⚙️ RPM:', layout=Layout(width='550px'), max=2500, min=500, step=10, style=S…

FloatSlider(value=50.0, description='🔩 Torque:', layout=Layout(width='550px'), min=10.0, step=1.0, style=Slide…

Output()